# Hakam: a distilled VLM on Sports-QA ⚽

We trained a 0.8B model to match a 27B model on sports video question answering. Same test accuracy (39.9% vs 39.7%, n=1000), 30× smaller. This notebook runs the full demo on Colab's **free T4 GPU**, with answers in a few seconds.

| model | params | accuracy |
|---|---:|---:|
| Qwen3.5-VL 0.8B, plain | 0.8B | 22.7% |
| **0.8B + our distilled LoRA** | **0.8B** | **39.9%** |
| Qwen3.5-VL 27B (the teacher) | 27B | 39.7% |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. The last cell prints a public `gradio.live` link. Open it, click a match replay, ask a question.

Code & evaluation: [github.com/ANANAS-HA/HAKAM](https://github.com/ANANAS-HA/HAKAM) · Adapter: [ananas0/qwen3.5-0.8b-sportsqa-distill-lora](https://huggingface.co/ananas0/qwen3.5-0.8b-sportsqa-distill-lora)

*Note: the published accuracy was measured in bf16. Colab's T4 doesn't support bf16, so this notebook falls back to fp16 there; answers are equivalent in practice.*


In [ ]:
# 1. Install dependencies (~2 min)
# Colab preinstalls an old torchao (0.10) that peft refuses; we don't use
# torchao, so removing it is the clean fix.
!pip uninstall -q -y torchao
!pip install -q "transformers>=5.5,<6" accelerate "peft>=0.19" qwen-vl-utils decord av gradio imageio-ffmpeg

In [ ]:
# 2. Download the demo clips and copy them out of the HF cache.
# (HF snapshot files are symlinks; Gradio blocks symlinked paths that resolve
#  outside allowed_paths, so we materialise real files in ./hakam_clips/.)
# worldcup/ is copied recursively: one subfolder per processed match.
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

_snap = Path(snapshot_download("ananas0/hakam-demo-clips", repo_type="dataset"))
LOCAL = Path.cwd() / "hakam_clips"
DATASET_CLIPS = LOCAL / "dataset"
WORLDCUP_CLIPS = LOCAL / "worldcup"

def _copy_tree(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        if item.is_dir():
            _copy_tree(item, dst / item.name)
        elif item.suffix.lower() in (".mp4", ".json", ".png"):
            shutil.copy(item, dst / item.name)   # follows symlinks -> real file

for sub in ("dataset", "worldcup", "assets"):
    if (_snap / sub).exists():
        _copy_tree(_snap / sub, LOCAL / sub)
    else:
        (LOCAL / sub).mkdir(parents=True, exist_ok=True)

print("dataset clips  :", sorted(p.name for p in DATASET_CLIPS.glob("*.mp4")))
print("worldcup match folders:",
      sorted(d.name for d in WORLDCUP_CLIPS.iterdir() if d.is_dir()) or "(none yet)")


In [ ]:
# 3. Load the model: Qwen3.5-0.8B base + the distilled LoRA adapter (from the Hub)
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

assert torch.cuda.is_available(), "Enable the GPU runtime: Runtime -> Change runtime type -> T4 GPU"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("device: cuda, dtype:", DTYPE)

BASE_ID = "Qwen/Qwen3.5-0.8B"
ADAPTER_ID = "ananas0/qwen3.5-0.8b-sportsqa-distill-lora"

processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_ID, dtype=DTYPE, device_map="cuda", trust_remote_code=True)
model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()
print("ready. VRAM: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))


In [ ]:
# 4. The demo app, using the same locked configuration as the published evaluation
import json, time
import gradio as gr
from qwen_vl_utils import process_vision_info

FPS = 2.0
SYSTEM_PROMPT = (
    "You are an elite sports video analyst with deep expertise across basketball, football, "
    "volleyball, and gymnastics (including aerobic gymnastics, vault, uneven bars, balance beam, "
    "and floor exercise). You watch short game or performance clips and parse them the way a "
    "coach or color commentator would: identifying the sport, naming specific techniques and "
    "skills (e.g. 2-point shot, spike, push-up, split, round-off, flic-flac, giant circle), "
    "counting discrete events and the number of athletes involved, tracking temporal order "
    "(what comes before / after what), and reasoning about cause and effect (why an action "
    "succeeded or failed, what a counterfactual outcome would have been).\n\n"
    "When given a question, think it through step by step — describe what you see in the clip, "
    "locate the moment the question refers to, and then give a precise, expert answer. Be "
    "decisive: sports are concrete, so give committed answers rather than hedged ones. "
    "If the question is a yes / no, still justify briefly. If it asks 'how many', count carefully. "
    "If it asks for the name of an action, use the technical term.\n\n"
    "**Outcome questions must be answered from the action itself, not from interface elements.** "
    "When a question asks whether an action *succeeded* (did the shot go in, did the team score, "
    "did the spike land, did the dismount stick), your verdict must come from observing the action's "
    "physical result: ball relative to rim/net, body relative to landing surface, ball crossing the "
    "goal line. **Do not use scoreboards, scorelines, point counters, or referee gestures as evidence "
    "of success or failure.** Scoreboards are graphical UI, lag the action by 1-3 seconds, and may not "
    "update within the clip's duration. If the play is cut off before the result is visually resolved, "
    "say so explicitly and answer from what was visible — do not infer success or failure from a static "
    "score.\n\n"
    "**Be concise.** Aim for around 300-500 tokens total. Identify the moment, state the evidence, "
    "commit to a verdict. Do not re-examine the same evidence twice or hedge between interpretations. "
    "Brevity over rumination."
)

def build_messages(video_path, question):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "video", "video": str(video_path), "fps": FPS,
             "max_frames": 32, "max_pixels": 256 * 28 * 28},
            {"type": "text", "text": question},
        ]},
    ]

def answer_question(video_path, question, history, temperature, top_p, top_k, max_new_tokens):
    history = history or []
    if not video_path:
        history += [{"role": "user", "content": question},
                    {"role": "assistant", "content": "Pick a replay from the gallery or upload a clip first."}]
        return history, ""
    if not question or not question.strip():
        return history, ""
    msgs = build_messages(video_path, question)
    text = processor.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    try:
        _, video_inputs = process_vision_info(msgs)
        inputs = processor(text=[text], images=None, videos=video_inputs,
                           padding=True, return_tensors="pt").to("cuda")
        plen = inputs["input_ids"].shape[1]
        t0 = time.perf_counter()
        with torch.inference_mode():
            gen = model.generate(**inputs, max_new_tokens=int(max_new_tokens),
                                 do_sample=True, temperature=float(temperature),
                                 top_p=float(top_p), top_k=int(top_k), min_p=0.0,
                                 repetition_penalty=1.1)
        dt = time.perf_counter() - t0
        toks = gen[0, plen:].cpu()
        out = processor.batch_decode([toks], skip_special_tokens=True,
                                     clean_up_tokenization_spaces=False)[0].strip()
        out += f"\n\n*({len(toks)} tokens in {dt:.1f}s)*"
    except Exception as e:
        out = f"generation error: {type(e).__name__}: {e}"
    history += [{"role": "user", "content": question},
                {"role": "assistant", "content": out}]
    return history, ""

# ---- clip inventory (two-level: World Cup matches + test-set samples) ----
# Same paths cell 2 uses; redefined here so this cell never depends on
# cell 2's variables (only on the files it downloaded).
from pathlib import Path
LOCAL = Path.cwd() / "hakam_clips"
DATASET_CLIPS = LOCAL / "dataset"
WORLDCUP_CLIPS = LOCAL / "worldcup"

K_SLOTS = 12
SRC_WC = "World Cup 2026 Match Highlights"
SRC_TS = "Test Set Samples"

# First four match the Sports-QA training distribution (best answers);
# the rest are the questions fans actually ask about match highlights.
WC_QUESTIONS = [
    "Which team scores in the video?",
    "Do the athletes shoot successfully?",
    "What is the effect of the shoot?",
    "What is the video about?",
    "Who scores the goal?",
    "How does the player score the goal?",
    "Does the goalkeeper save the shot?",
    "Is the goal scored from a penalty?",
    "What happens after the goal is scored?",
    "How many attackers are involved in the play?",
]

# ---- Hakam visual identity (matches the GP2 poster) ----
ASSETS = LOCAL / "assets"

_HEX = ("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' "
        "width='56' height='97' viewBox='0 0 56 97'%3E%3Cpath d='M28 0l28 16v32"
        "L28 64 0 48V16zM28 64l28 16v17M28 64L0 80v17' fill='none' "
        "stroke='%2322c55e' stroke-opacity='0.07'/%3E%3C/svg%3E")

HAKAM_CSS = f"""
.gradio-container {{
  background:
    url("{_HEX}"),
    radial-gradient(900px 420px at 15% -5%, rgba(34,197,94,.10), transparent 60%),
    radial-gradient(900px 420px at 85% -5%, rgba(34,197,94,.06), transparent 60%),
    #0a0e0a !important;
}}
#hero {{
  border-top: 4px solid #22c55e;
  background: linear-gradient(180deg, rgba(22,163,74,.16), rgba(10,14,10,0) 70%);
  border-radius: 12px;
  padding: 22px 26px 16px;
  margin-bottom: 6px;
}}
#hero .hero-row {{ display: flex; align-items: center; gap: 22px; }}
#hero .badge {{
  width: 150px; height: 150px; border-radius: 50%; flex: none;
  border: 3px solid #22c55e; background: #06130a;
  display: flex; align-items: center; justify-content: center;
  font-size: 44px; box-shadow: 0 0 24px rgba(34,197,94,.35);
  overflow: hidden;
}}
#hero .badge img {{ width: 100%; height: 100%; object-fit: cover; }}
#hero .badge.has-img {{ border: none; background: transparent; }}
#hero .eyebrow {{
  color: #4ade80; letter-spacing: .38em; text-transform: uppercase;
  font-size: .72rem; font-weight: 700; margin-bottom: 2px;
}}
#hero h1 {{
  font-family: 'Archivo Black', 'Archivo', sans-serif;
  font-size: 3.1rem; line-height: 1.02; color: #fff;
  letter-spacing: .02em; margin: 0 0 4px;
}}
#hero .sub {{ color: #e5e7eb; font-size: 1.05rem; margin: 2px 0; }}
#hero .sub .accent {{ color: #4ade80; font-weight: 600; }}
#hero .pitch {{ color: #9ca3af; font-size: .9rem; margin: 2px 0 10px; }}
#hero .pitch .accent {{ color: #fbbf24; font-weight: 700; }}
#hero .chip {{
  display: inline-block; border: 1px solid rgba(34,197,94,.45);
  background: rgba(22,101,52,.22); color: #bbf7d0;
  padding: 3px 12px; border-radius: 999px;
  font-size: .78rem; font-weight: 600; margin: 0 6px 4px 0;
}}
.q-wrap {{ flex-wrap: wrap !important; }}
.q-wrap button {{ flex: 0 1 auto !important; min-width: 0 !important; }}
.section-bar {{
  background: linear-gradient(90deg, #16a34a, #052e16) !important;
  border-radius: 8px !important; padding: 6px 14px !important;
}}
.section-bar * {{ color: #fff !important; font-weight: 700 !important; }}
#footer {{
  margin-top: 10px; padding: 12px 8px 4px; text-align: center;
  border-top: 1px solid rgba(34,197,94,.2);
  color: #6b7280; font-size: .8rem;
}}
#footer a {{ color: #4ade80; text-decoration: none; }}
"""

HAKAM_HEAD = (
    "<link rel='preconnect' href='https://fonts.googleapis.com'>"
    "<link href='https://fonts.googleapis.com/css2?family=Archivo+Black&"
    "family=Archivo:wght@400;600;700&display=swap' rel='stylesheet'>"
)

def hakam_theme():
    # Poster palette baked into the theme itself (always dark, robust in iframes)
    return gr.themes.Base(
        primary_hue=gr.themes.colors.green,
        neutral_hue=gr.themes.colors.gray,
        font=[gr.themes.GoogleFont("Archivo"), "ui-sans-serif", "system-ui"],
        font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace"],
    ).set(
        body_background_fill="#0a0e0a",
        body_text_color="#d1d5db",
        body_text_color_subdued="#9ca3af",
        background_fill_primary="#10150f",
        background_fill_secondary="#0d120d",
        block_background_fill="#10150f",
        block_border_color="rgba(34,197,94,0.18)",
        block_label_background_fill="#0f2417",
        block_label_text_color="#86efac",
        block_title_text_color="#e5e7eb",
        border_color_primary="rgba(34,197,94,0.28)",
        input_background_fill="#0d120d",
        input_border_color="rgba(34,197,94,0.25)",
        button_primary_background_fill="linear-gradient(90deg,#16a34a,#15803d)",
        button_primary_background_fill_hover="linear-gradient(90deg,#22c55e,#16a34a)",
        button_primary_text_color="#ffffff",
        button_secondary_background_fill="#182018",
        button_secondary_background_fill_hover="#1f2b1f",
        button_secondary_text_color="#d1d5db",
        color_accent_soft="#0f2417",
    )

def hero_html():
    logo_png = ASSETS / "logo.png"
    if logo_png.exists():
        import base64
        b64 = base64.b64encode(logo_png.read_bytes()).decode()
        badge = f"<img src='data:image/png;base64,{b64}' alt='Hakam logo'>"
        badge_cls = "badge has-img"   # the logo brings its own green ring
    else:
        badge, badge_cls = "⚽", "badge"
    return f"""
<div class="hero-row">
  <div class="{badge_cls}">{badge}</div>
  <div>
    <div class="eyebrow">Graduation Project · Sports AI</div>
    <h1>HAKAM</h1>
    <p class="sub">A Distilled <span class="accent">Vision-Language Model</span>
       for Sports Video Question Answering</p>
    <p class="pitch"><span class="accent">30× compression</span> of a 27B VLM
       into a 0.8B student that matches its accuracy on a single consumer GPU.</p>
    <div>
      <span class="chip">0.8B student</span>
      <span class="chip">39.9% vs 39.7% teacher</span>
      <span class="chip">30× smaller</span>
      <span class="chip">~2 GB VRAM</span>
    </div>
  </div>
</div>"""

FOOTER_HTML = (
    "<div>HAKAM · CS Graduation Project 2 · IMSIU CCIS &nbsp;·&nbsp; "
    "<a href='https://huggingface.co/ananas0/qwen3.5-0.8b-sportsqa-distill-lora' "
    "target='_blank'>Model card</a> · "
    "<a href='https://github.com/ANANAS-HA/HAKAM' target='_blank'>GitHub</a></div>"
)

# knockout stages first, then group games alphabetically
STAGE_ORDER = ["Final", "Third-Place", "Semi-Final", "Semi-Finals",
               "Quarter-Final", "Quarter-Finals", "Round of 16", "Round_of_16"]

def stage_rank(folder):
    stage = folder.split("__", 1)[1] if "__" in folder else ""
    for i, tok in enumerate(STAGE_ORDER):
        if stage.startswith(tok):
            return (i, folder)
    return (len(STAGE_ORDER), folder)

manifest = json.loads((DATASET_CLIPS / "manifest.json").read_text()) \
           if (DATASET_CLIPS / "manifest.json").exists() else []
dataset_clips = [{"path": DATASET_CLIPS / e["file"],
                  "question": e.get("suggested_question", "What is the video about?")}
                 for e in manifest if (DATASET_CLIPS / e["file"]).exists()] or \
                [{"path": p, "question": "What is the video about?"}
                 for p in sorted(DATASET_CLIPS.glob("*.mp4"))]

def list_matches():
    if not WORLDCUP_CLIPS.exists():
        return []
    folders = [d.name for d in WORLDCUP_CLIPS.iterdir()
               if d.is_dir() and any(d.glob("*.mp4"))]
    return sorted(folders, key=stage_rank)

def display_name(folder):
    return folder.replace("__", " — ").replace("_", " ")

def folder_from_display(display, folders):
    for f in folders:
        if display_name(f) == display:
            return f
    return None

def match_segments(folder):
    return sorted((WORLDCUP_CLIPS / folder).glob("*.mp4"))

match_folders = list_matches()
wc_default = bool(match_folders)

unique_questions = []
for c in dataset_clips:
    if c["question"] not in unique_questions:
        unique_questions.append(c["question"])

with gr.Blocks(title="Hakam: a distilled VLM on Sports-QA") as demo:
    gr.HTML(hero_html(), elem_id="hero")
    gr.Markdown("Pick a clip source, load a clip, ask what happened. "
                "Sweet spot: **8–16 s clips, one action each**.")
    with gr.Row():
        with gr.Column(scale=1, min_width=340):
            video = gr.Video(label="Selected clip", sources=["upload"])
            source = gr.Radio(choices=[SRC_WC, SRC_TS],
                              value=SRC_WC if wc_default else SRC_TS,
                              label="Clip source")

            with gr.Group(visible=wc_default) as wc_group:
                match_sel = gr.Dropdown(
                    choices=[display_name(f) for f in match_folders],
                    value=None, label="Match", interactive=True)
                refresh_btn = gr.Button("↻ Refresh match list", size="sm")
                wc_note = gr.Markdown("*No processed matches yet. Check back soon.*",
                                      visible=not match_folders)
                gr.Markdown("Watch a segment, then press **Use** to load it:",
                            elem_classes=["section-bar"])
                slot_videos, slot_btns = [], []
                for i in range(K_SLOTS):
                    sv = gr.Video(visible=False, interactive=False,
                                  height=150, label=f"Clip {i+1}")
                    sb = gr.Button(f"Use clip {i+1}", size="sm", visible=False)
                    slot_videos.append(sv); slot_btns.append(sb)

            with gr.Group(visible=not wc_default) as ts_group:
                gr.Markdown("Replays from the Sports-QA test set (click to load)",
                        elem_classes=["section-bar"])
                gr.Examples(examples=[[str(c["path"])] for c in dataset_clips],
                            inputs=[video], fn=lambda v: v, outputs=[video],
                            run_on_click=True, cache_examples=False,
                            label=None, examples_per_page=5)

        with gr.Column(scale=2):
            chat = gr.Chatbot(label="Q&A", height=420)
            wc_prompt_btns, ts_prompt_btns = [], []
            with gr.Group():
                gr.Markdown("**Suggested questions** (click to fill the box):",
                            elem_classes=["section-bar"])
                with gr.Row(visible=wc_default,
                            elem_classes=["q-wrap"]) as wc_prompt_row:
                    for q in WC_QUESTIONS:
                        wc_prompt_btns.append(gr.Button(q, size="sm"))
                with gr.Row(visible=not wc_default,
                            elem_classes=["q-wrap"]) as ts_prompt_row:
                    for q in unique_questions:
                        ts_prompt_btns.append(gr.Button(q, size="sm"))
            question = gr.Textbox(label="Question",
                                  placeholder="e.g. Which team scores in the video?",
                                  lines=2)
            with gr.Row():
                send = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear chat")
    with gr.Accordion("Advanced sampling settings", open=False):
        temperature = gr.Slider(0.0, 1.5, value=0.5, step=0.05, label="temperature")
        top_p = gr.Slider(0.0, 1.0, value=0.9, step=0.05, label="top_p")
        top_k = gr.Slider(1, 100, value=20, step=1, label="top_k")
        max_new = gr.Slider(16, 512, value=256, step=16, label="max_new_tokens")

    # ---- wiring (validated in _dev_gallery_ui.py with real WC clips) ----
    def on_source_change(src):
        wc = src == SRC_WC
        return (gr.update(visible=wc), gr.update(visible=not wc),
                gr.update(visible=wc), gr.update(visible=not wc))
    source.change(on_source_change, inputs=source,
                  outputs=[wc_group, ts_group, wc_prompt_row, ts_prompt_row])

    def on_match_change(display):
        folders = list_matches()
        folder = folder_from_display(display, folders) if display else None
        segs = match_segments(folder) if folder else []
        vids = [gr.update(value=str(segs[i]), visible=True,
                          label=f"Clip {i+1} of {len(segs)}")
                if i < len(segs) else gr.update(value=None, visible=False)
                for i in range(K_SLOTS)]
        btns = [gr.update(visible=i < len(segs)) for i in range(K_SLOTS)]
        return vids + btns
    match_sel.change(on_match_change, inputs=match_sel,
                     outputs=slot_videos + slot_btns)

    def on_refresh():
        folders = list_matches()
        return (gr.update(choices=[display_name(f) for f in folders], value=None),
                gr.update(visible=not folders))
    refresh_btn.click(on_refresh, inputs=[], outputs=[match_sel, wc_note])

    for sv, sb in zip(slot_videos, slot_btns):
        sb.click(lambda v: v, inputs=[sv], outputs=video)

    for qb, qtext in zip(wc_prompt_btns + ts_prompt_btns,
                         WC_QUESTIONS + unique_questions):
        qb.click(lambda q=qtext: q, inputs=[], outputs=question)

    send.click(answer_question,
               [video, question, chat, temperature, top_p, top_k, max_new],
               [chat, question])
    question.submit(answer_question,
                    [video, question, chat, temperature, top_p, top_k, max_new],
                    [chat, question])
    clear.click(lambda: [], [], [chat])
    gr.HTML(FOOTER_HTML, elem_id="footer")

demo.launch(share=True, allowed_paths=[str(LOCAL)],
            theme=hakam_theme(), css=HAKAM_CSS, head=HAKAM_HEAD)
